In [1]:
# 1. Actualizamos los repositorios del sistema e instalamos Java 17
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

# 2. Instalamos PySpark
!pip install -q pyspark

import os

# 3. Configuramos la variable de entorno para Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from pyspark import SparkContext

from pyspark import SparkContext
from pyspark.sql import SQLContext, Row
from pyspark.sql import Window
from pyspark.sql import functions as F
sc = SparkContext("local", "TrabajoPractico")
root_path = '/content/'

##Enunciado
La ciudad de Zootopia almacena la geoposición de todos los vehículos que transitan durante el
día.


##El dataset está conformado por:
• ID_Vehiculo
• Latitud
• Longitud
• Timestamp
• Destino

##1) Implemente un script de pySpark que permita conocer cuántos viajes realizó cada vehículo.  Recordar que un viaje es una serie de coordenadas que finalizan en un destino determinado.

Ejemplo:

Vehículo  Viajes


  511           4
  
697            11

151           1

653           3


In [3]:

viajes = sc.textFile("dataset.txt")

viajes = viajes.filter(lambda t: len(t.split()) >= 5)

viajes = viajes.map(lambda t: (t.split()[0], 1))

resultados = viajes.reduceByKey(lambda a, b: a + b)

resultados_collect = resultados.collect()

for resultado in resultados_collect:
  print(f"Vehículo {resultado[0]} ha realizado {resultado[1]} viajes.")



Vehículo 75 ha realizado 2 viajes.
Vehículo 303 ha realizado 3 viajes.
Vehículo 615 ha realizado 2 viajes.
Vehículo 405 ha realizado 2 viajes.
Vehículo 312 ha realizado 1 viajes.
Vehículo 174 ha realizado 4 viajes.
Vehículo 194 ha realizado 2 viajes.
Vehículo 133 ha realizado 3 viajes.
Vehículo 818 ha realizado 1 viajes.
Vehículo 166 ha realizado 4 viajes.
Vehículo 553 ha realizado 1 viajes.
Vehículo 454 ha realizado 1 viajes.
Vehículo 353 ha realizado 1 viajes.
Vehículo 47 ha realizado 2 viajes.
Vehículo 201 ha realizado 1 viajes.
Vehículo 169 ha realizado 2 viajes.
Vehículo 385 ha realizado 6 viajes.
Vehículo 215 ha realizado 2 viajes.
Vehículo 497 ha realizado 2 viajes.
Vehículo 295 ha realizado 2 viajes.
Vehículo 281 ha realizado 2 viajes.
Vehículo 334 ha realizado 1 viajes.
Vehículo 346 ha realizado 1 viajes.
Vehículo 808 ha realizado 5 viajes.
Vehículo 29 ha realizado 3 viajes.
Vehículo 77 ha realizado 1 viajes.
Vehículo 890 ha realizado 3 viajes.
Vehículo 480 ha realizado 1 viaj

##2) Implemente un script de Spark que permita conocer cual es el top 3 de los "tipos" de destinos más visitados. NO interesa contar a los destinos "Otro".

Ejemplo:

Tipo de destino  Cantidad de visitas

Supermercado          316  

Museo                  197

Escuela                  193


In [4]:
viajes = sc.textFile("dataset.txt")

sqlContext = SQLContext(sc)

viajes = viajes.map(lambda l:
                    Row(
                        id_vehiculo = l.split()[0],
                        latitud = l.split()[1],
                        longitud = l.split()[2],
                        timestamp = l.split()[3],
                        destino = l.split()[4] if len(l.split()) >= 5 else ""
                    ))

viajes = sqlContext.createDataFrame(viajes)

viajes = viajes.filter(viajes.destino != "Otro")

tiposDestinos = Window.partitionBy("destino")

resultados = viajes.withColumn("cantidad", F.count("destino").over(tiposDestinos))

ranking = Window.orderBy(F.desc("cantidad"))

resultados = resultados.dropDuplicates(["destino"])

resultados = resultados.withColumn("rank", F.row_number().over(ranking))

resultados.select("destino", "cantidad").where(resultados.rank <= 3).show()



/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


+------------+--------+
|     destino|cantidad|
+------------+--------+
|            |   85025|
|Supermercado|     165|
|     Escuela|      98|
+------------+--------+



##3) Implemente un script de Spark que permita conocer la cantidad de vehículos (distintos) en movimiento por franja horaria. Un vehículo se considera en movimiento dentro de una franja horario si al menos registra N geolocalizaciones dentro de la propia franja. La duración de la franja horaria y el N son parámetros de la consulta.


Ejemplo:


Franja horaria  Vehículos en movimiento


0 - 3000      843

3000 - 6000       673

6000 - 9000        449

9000 - 12000         248


In [5]:
viajes = sc.textFile("dataset.txt")

franjas = [(0, 3000), (3000, 6000), (6000, 9000), (9000, 12000)]

cantGeolocalizaciones = 1000
def porNodo(acumulador,x):

  franja1,franja2,franja3,franja4 = acumulador

  timestamp = int(x.split()[3])

  if timestamp in range(franjas[0][0],franjas[0][1]):
    franja1 += 1
  elif timestamp in range(franjas[1][0],franjas[1][1]):
    franja2 += 1
  elif timestamp in range(franjas[2][0],franjas[2][1]):
    franja3 += 1
  elif timestamp in range(franjas[3][0],franjas[3][1]):
    franja4 += 1

  return (franja1,franja2,franja3,franja4)


def porTotal(acumulador1,acumulador2):

  return (acumulador1[0]+acumulador2[0],
          acumulador1[1]+acumulador2[1],
          acumulador1[2]+acumulador2[2],
          acumulador1[3]+acumulador2[3]
          )


iniciales = (0,0,0,0)
viajes = viajes.aggregate(iniciales,porNodo,porTotal)

for i in range(len(franjas)):
  print("franja:",i,f"{franjas[i][0]}-{franjas[i][1]} {viajes[i]}")




franja: 0 0-3000 49196
franja: 1 3000-6000 22391
franja: 2 6000-9000 9339
franja: 3 9000-12000 3480


In [6]:
viajes = sc.textFile("dataset.txt")

franjas = [(0, 3000), (3000, 6000), (6000, 9000), (9000, 12000)]
cantGeolocalizaciones = 1


def porNodo(acumulador, x):
  franja1, franja2, franja3, franja4 = acumulador
  timestamp = int(x)

  if timestamp in range(franjas[0][0], franjas[0][1]):
    franja1 += 1
  elif timestamp in range(franjas[1][0], franjas[1][1]):
    franja2 += 1
  elif timestamp in range(franjas[2][0], franjas[2][1]):
    franja3 += 1
  elif timestamp in range(franjas[3][0], franjas[3][1]):
    franja4 += 1

  return (franja1, franja2, franja3, franja4)


def porTotal(acumulador1, acumulador2):
  return (
      acumulador1[0] + acumulador2[0],
      acumulador1[1] + acumulador2[1],
      acumulador1[2] + acumulador2[2],
      acumulador1[3] + acumulador2[3],
  )

viajes = viajes.map(lambda linea: (linea.split()[0], linea.split()[3]))

iniciales = (0, 0, 0, 0)


viajes_agrupados = viajes.aggregateByKey(iniciales, porNodo, porTotal)

final = viajes_agrupados.aggregate(
    (0, 0, 0, 0),
    lambda res, ori: (
        res[0] + 1 if ori[1][0] >= cantGeolocalizaciones else res[0],
        res[1] + 1 if ori[1][1] >= cantGeolocalizaciones else res[1],
        res[2] + 1 if ori[1][2] >= cantGeolocalizaciones else res[2],
        res[3] + 1 if ori[1][3] >= cantGeolocalizaciones else res[3],
    ),
    lambda r1, r2: (
        r1[0] + r2[0],
        r1[1] + r2[1],
        r1[2] + r2[2],
        r1[3] + r2[3],
    ),
)

for i in range(len(franjas)):
  print("franja:",i,f"{franjas[i][0]}-{franjas[i][1]} {final[i]}")

franja: 0 0-3000 693
franja: 1 3000-6000 410
franja: 2 6000-9000 190
franja: 3 9000-12000 80


##4) Implemente un script de Spark que permita conocer la velocidad promedio de cada vehículo: el promedio de las velocidades de todos sus viajes. La velocidad de un viaje se calcula como el tiempo del viaje (timestamp de fin – timestamp de inicio) dividido la cantidad de cuadras del viaje.


Ejemplo:


Vehículo Velocidad promedio


511 23.6

697 11.5


151 101


653 32.4

In [17]:
import os


def generarID():
  return os.urandom(16).hex()


def asignarIDviaje(lineas):
  # Ordenamos cronológicamente por timestamp (en realidad no me cambia nada el resultado
  #si lo agrego o no, pero intuyo que en un entorno real al tener varios nodos si pueda
  #llegar a ser necesario. Lo dejo nomas para tenerlo en cuenta )
  lineas_ordenadas = sorted(lineas, key=lambda x: (float(x[2]), x[3] != ""))

  resultados = []
  id_viaje = generarID()

  for l in lineas_ordenadas:
    latitud = l[0]
    longitud = l[1]
    timestamp = l[2]
    destino = l[3]

    # Guardamos el punto actual con el ID del viaje en curso
    resultados.append((latitud, longitud, timestamp, destino, id_viaje))

    # Si la fila tiene destino, implica que el viaje termino.
    # Generamos un nuevo ID para el próximo viaje.
    if destino != "":
      id_viaje = generarID()

  return resultados


viajes = sc.textFile("dataset.txt")

viajes = viajes.map(
    lambda l: (
        l.split()[0],  # ID_Vehiculo (Clave)
        (
            l.split()[1],  # Latitud
            l.split()[2],  # Longitud
            l.split()[3],  # Timestamp
            l.split()[4] if len(l.split()) >= 5 else "",  # Destino
        ),
    )
)

# Agrupamos por vehículo
viajes = viajes.groupByKey()

viajes = viajes.mapValues(asignarIDviaje)

# creo una clave compuesta: (ID_Vehiculo, ID_Viaje)
viajes_por_trayecto = viajes.flatMap(lambda x: [
    (
        (x[0], fila[4]),  #  (Vehículo, ID_Viaje)
        (fila[0], fila[1], fila[2], fila[3]),  # latitud,longitud,timestamp, destino
    )
    for fila in x[1]
])

#  Agrupo por trayecto (Vehículo, Viaje)
viajes_por_trayecto = viajes_por_trayecto.groupByKey()


# 5. Función para calcular la velocidad de cada viaje individual
def calcular_velocidad_viaje(filas):
  timestamps = [float(f[2]) for f in filas] ##lista de todos los timestamps para cada correspondiente tupla viaje-vehiculo
  min_ts = min(timestamps)
  max_ts = max(timestamps)
  cantidad_cuadras = len(filas)

  return (
      (max_ts - min_ts) / cantidad_cuadras if cantidad_cuadras > 0 else 0.0
  )


velocidades_por_viaje = viajes_por_trayecto.mapValues(calcular_velocidad_viaje)

#estructura resultante = ( (id_vehiculo,id_viaje) , (prom_viaje))


#Reestructuro las tuplas para poder usar aggregate y obtener los valores finales, dejo a id_vehiculo como clave
por_vehiculo = velocidades_por_viaje.map(
    lambda x: (x[0][0], (x[1], 1))  # (ID_Vehiculo, (velocidad_viaje, 1)) agrego el 1 como valor para lugar ir obteniendo la cantidad total de viajes para sacar el promedio total de todos los viajes
)

promedios_vehiculo = por_vehiculo.aggregateByKey(
    (0.0, 0),  # (Suma de velocidades, Cantidad de viajes)
    lambda res, v: (res[0] + v[0], res[1] + v[1]),
    lambda res1, res2: (res1[0] + res2[0], res1[1] + res2[1]),
)


resultado_final = promedios_vehiculo.map(
    lambda x: (int(x[0]), x[1][0] / x[1][1] if x[1][1] > 0 else 0.0)
)


resultado_final = resultado_final.sortBy(lambda x: x[0]).collect() #el sortBy es opcional, no me lo pidieron en el enunciado
#pero queda mejor presentado


for vehiculo, vel_prom in resultado_final:
  print(f"Vehículo {vehiculo}: Velocidad promedio = {vel_prom:.2f}")




Vehículo 1: Velocidad promedio = 16.49
Vehículo 2: Velocidad promedio = 5.82
Vehículo 3: Velocidad promedio = 23.31
Vehículo 4: Velocidad promedio = 52.97
Vehículo 5: Velocidad promedio = 9.84
Vehículo 6: Velocidad promedio = 54.29
Vehículo 10: Velocidad promedio = 26.65
Vehículo 11: Velocidad promedio = 3.39
Vehículo 12: Velocidad promedio = 24.84
Vehículo 13: Velocidad promedio = 30.43
Vehículo 14: Velocidad promedio = 1.93
Vehículo 16: Velocidad promedio = 2.70
Vehículo 17: Velocidad promedio = 47.54
Vehículo 18: Velocidad promedio = 22.65
Vehículo 19: Velocidad promedio = 22.83
Vehículo 20: Velocidad promedio = 29.45
Vehículo 21: Velocidad promedio = 27.30
Vehículo 22: Velocidad promedio = 49.89
Vehículo 23: Velocidad promedio = 42.51
Vehículo 24: Velocidad promedio = 49.10
Vehículo 26: Velocidad promedio = 36.70
Vehículo 27: Velocidad promedio = 23.87
Vehículo 28: Velocidad promedio = 50.11
Vehículo 29: Velocidad promedio = 31.89
Vehículo 30: Velocidad promedio = 58.62
Vehículo 31